 **Load existing Silver dependencies**

Reusing `silver_orders` and `silver_reviews` (already built in `02_silver_orders_reviews`) as the source of truth for foreign key validation, rather than re-reading Bronze.

In [1]:
df_orders_silver = spark.table("silver_orders")
df_reviews_silver = spark.table("silver_reviews")

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 3, Finished, Available, Finished, False)

**Reusable Bronze → Silver cleaning function**

A single, parameterized function handles the row-level tables (customers, sellers, products, order_items, payments): load raw CSV, cast declared columns to proper types, check declared foreign keys against reference tables, and write the result as a Delta table.

Geolocation is intentionally **not** run through this function — it needs aggregation to zip-code centroids, not row-level casting/validation, so forcing it into this abstraction would blur the function's contract. It's handled separately below.

In [2]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import IntegerType, DoubleType

def clean_and_write_table(
    table_name,
    source_file,
    timestamp_cols=None,
    int_cols=None,
    double_cols=None,
    fk_checks=None,   # list of (fk_column, reference_df, reference_column)
    write=True
):
    """
    Generic Bronze -> Silver cleaner: casts types and validates foreign keys.
    Returns the cleaned dataframe and prints a findings summary.
    """
    timestamp_cols = timestamp_cols or []
    int_cols = int_cols or []
    double_cols = double_cols or []
    fk_checks = fk_checks or []

    df = spark.read.option("header", "true").csv(f"Files/bronze/olist/{source_file}")
    raw_count = df.count()

    for c in timestamp_cols:
        df = df.withColumn(c, to_timestamp(col(c)))
    for c in int_cols:
        df = df.withColumn(c, col(c).cast(IntegerType()))
    for c in double_cols:
        df = df.withColumn(c, col(c).cast(DoubleType()))

    print(f"=== {table_name} ===")
    print(f"Raw row count: {raw_count}")

    for fk_col, ref_df, ref_col in fk_checks:
        orphan_count = df.join(ref_df.select(ref_col), df[fk_col] == ref_df[ref_col], "left_anti").count()
        print(f"Orphaned {fk_col} refs (no matching {ref_col}): {orphan_count}")

    if write:
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Written to Delta table: {table_name}\n")

    return df

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 4, Finished, Available, Finished, False)

**Customers, sellers, category translation**

These three are simple lookup/dimension tables with no type-casting or FK issues expected — running them through the function mainly confirms row counts and gives a consistent write pattern.

In [3]:
df_customers_silver = clean_and_write_table(
    table_name="silver_customers",
    source_file="olist_customers_dataset.csv"
)

df_sellers_silver = clean_and_write_table(
    table_name="silver_sellers",
    source_file="olist_sellers_dataset.csv"
)

df_category_silver = clean_and_write_table(
    table_name="silver_category_translation",
    source_file="product_category_name_translation.csv"
)

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 5, Finished, Available, Finished, False)

=== silver_customers ===
Raw row count: 99441
Written to Delta table: silver_customers

=== silver_sellers ===
Raw row count: 3095
Written to Delta table: silver_sellers

=== silver_category_translation ===
Raw row count: 71
Written to Delta table: silver_category_translation



**Products**

Several numeric columns (`product_weight_g`, `product_length_cm`, etc.) load as strings by default and need explicit casting. Also checking here whether any products have a null `product_category_name` — a known issue in the raw Olist data, not a transcription error.

In [4]:
df_products_silver = clean_and_write_table(
    table_name="silver_products",
    source_file="olist_products_dataset.csv",
    int_cols=["product_name_lenght", "product_description_lenght", "product_photos_qty"],
    double_cols=["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
)

print("Products missing category name:", df_products_silver.filter(col("product_category_name").isNull()).count())

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 6, Finished, Available, Finished, False)

=== silver_products ===
Raw row count: 32951
Written to Delta table: silver_products

Products missing category name: 610


**Order items**

Casts `price`, `freight_value`, and `shipping_limit_date`, and checks referential integrity against orders, products, and sellers — this table has three foreign keys, so it's the most likely place to find orphaned references beyond what we already found in reviews.

In [5]:
df_order_items_silver = clean_and_write_table(
    table_name="silver_order_items",
    source_file="olist_order_items_dataset.csv",
    timestamp_cols=["shipping_limit_date"],
    double_cols=["price", "freight_value"],
    fk_checks=[
        ("order_id", df_orders_silver, "order_id"),
        ("product_id", df_products_silver, "product_id"),
        ("seller_id", df_sellers_silver, "seller_id"),
    ]
)

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 7, Finished, Available, Finished, False)

=== silver_order_items ===
Raw row count: 112650
Orphaned order_id refs (no matching order_id): 0
Orphaned product_id refs (no matching product_id): 0
Orphaned seller_id refs (no matching seller_id): 0
Written to Delta table: silver_order_items



**Order payments**

Casts `payment_value` and `payment_installments`, and checks that every payment references a real order.

In [6]:
df_payments_silver = clean_and_write_table(
    table_name="silver_payments",
    source_file="olist_order_payments_dataset.csv",
    int_cols=["payment_installments"],
    double_cols=["payment_value"],
    fk_checks=[("order_id", df_orders_silver, "order_id")]
)

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 8, Finished, Available, Finished, False)

=== silver_payments ===
Raw row count: 103886
Orphaned order_id refs (no matching order_id): 0
Written to Delta table: silver_payments



**Geolocation (handled separately)**

~1,000,163 raw rows exist for only ~19,000 unique zip codes — expected duplication by design, not a defect (see `docs/data-quality-findings.md`). Rather than row-level cleaning, this aggregates lat/long to a single centroid per zip code, city, and state.

In [7]:
from pyspark.sql.functions import avg

df_geo_bronze = spark.read.option("header", "true").csv("Files/bronze/olist/olist_geolocation_dataset.csv")

df_geo_silver = (
    df_geo_bronze
    .withColumn("geolocation_lat", col("geolocation_lat").cast(DoubleType()))
    .withColumn("geolocation_lng", col("geolocation_lng").cast(DoubleType()))
    .groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(avg("geolocation_lat").alias("avg_lat"), avg("geolocation_lng").alias("avg_lng"))
)

print("Raw geolocation rows:", df_geo_bronze.count())
print("Aggregated geolocation rows (zip centroids):", df_geo_silver.count())

df_geo_silver.write.format("delta").mode("overwrite").saveAsTable("silver_geolocation")

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 9, Finished, Available, Finished, False)

Raw geolocation rows: 1000163
Aggregated geolocation rows (zip centroids): 27912


In [8]:
df_products_silver.filter(col("product_category_name").isNull()).select(
    "product_id", "product_weight_g", "product_length_cm", "product_photos_qty"
).show(10)

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 10, Finished, Available, Finished, False)

+--------------------+----------------+-----------------+------------------+
|          product_id|product_weight_g|product_length_cm|product_photos_qty|
+--------------------+----------------+-----------------+------------------+
|a41e356c76fab6633...|           650.0|             17.0|              NULL|
|d8dee61c2034d6d07...|           300.0|             16.0|              NULL|
|56139431d72cd51f1...|           200.0|             20.0|              NULL|
|46b48281eb6d663ce...|         18500.0|             41.0|              NULL|
|5fb61f482620cb672...|           300.0|             35.0|              NULL|
|e10758160da97891c...|          2200.0|             16.0|              NULL|
|39e3b9b12cd0bf8ee...|           300.0|             16.0|              NULL|
|794de06c32a626a56...|           300.0|             18.0|              NULL|
|7af3e2da474486a35...|           200.0|             22.0|              NULL|
|629beb8e7317703dc...|          1400.0|             25.0|              NULL|

In [9]:
null_category_df = df_products_silver.filter(col("product_category_name").isNull())
print("Total null-category products:", null_category_df.count())
print("Of those, also missing photos_qty:", null_category_df.filter(col("product_photos_qty").isNull()).count())
print("Of those, also missing weight:", null_category_df.filter(col("product_weight_g").isNull()).count())

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 11, Finished, Available, Finished, False)

Total null-category products: 610
Of those, also missing photos_qty: 610
Of those, also missing weight: 1


In [10]:
for t in ["silver_customers", "silver_sellers", "silver_category_translation",
          "silver_products", "silver_order_items", "silver_payments", "silver_geolocation"]:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {t}").collect()[0]["cnt"]
    print(f"{t}: {cnt}")

StatementMeta(, 383dda05-cdb6-4b44-b935-17f0de642281, 12, Finished, Available, Finished, False)

silver_customers: 99441
silver_sellers: 3095
silver_category_translation: 71
silver_products: 32951
silver_order_items: 112650
silver_payments: 103886
silver_geolocation: 27912
